In [64]:
def create_sensors():
    sensors = [
        {"id": 1, "location": "Living Room", "type": "motion", "current_value": None},
        {"id": 2, "location": "Front Door", "type": "door", "current_value": None},
        {"id": 3, "location": "Kitchen", "type": "temperature", "current_value": None},
        {"id": 4, "location": "Bedroom", "type": "smoke", "current_value": None},
    ]
    return sensors

system_mode = "AWAY"

sensors = create_sensors()
print(sensors)

[{'id': 1, 'location': 'Living Room', 'type': 'motion', 'current_value': None}, {'id': 2, 'location': 'Front Door', 'type': 'door', 'current_value': None}, {'id': 3, 'location': 'Kitchen', 'type': 'temperature', 'current_value': None}, {'id': 4, 'location': 'Bedroom', 'type': 'smoke', 'current_value': None}]


In [ ]:

FREEZE_TEMP = 35
HOT_TEMP = 95
COMFORT_MIN = 65
COMFORT_MAX = 75


def check_sensor(sensor, mode):
    value = sensor["current_value"]
    location = sensor["location"]
    time = datetime.datetime.now().strftime("%H:%M:%S")

    
    if sensor["type"] == "motion" or sensor["type"] == "door":
        if value == "YES" and mode == "AWAY":
            return {"time": time, "severity": "HIGH", "category": "SECURITY",
                    "message": location + " activity while in AWAY mode!"}

    
    if sensor["type"] == "smoke":
        if value == "YES":
            return {"time": time, "severity": "CRITICAL", "category": "SAFETY",
                    "message": "Smoke in " + location + "!"}

    
    if sensor["type"] == "temperature":
        if value < FREEZE_TEMP or value > HOT_TEMP:
            return {"time": time, "severity": "HIGH", "category": "SAFETY",
                    "message": location + " temperature is " + str(value) + "F!"}
        if value < COMFORT_MIN or value > COMFORT_MAX:
            if mode == "HOME":
                return {"time": time, "severity": "LOW", "category": "COMFORT",
                        "message": location + " temperature is " + str(value) + "F"}

    
    return None




print(check_sensor(door, "AWAY"))
print(check_sensor(door, "HOME"))

{'time': '17:38:22', 'severity': 'HIGH', 'category': 'SECURITY', 'message': 'Front Door activity while in AWAY mode!'}
None


In [57]:
event_log = []   # the list where logged alerts are saved


# 1. Generate a reading: use random to give a sensor a new value
def generate_reading(sensor):
    if sensor["type"] == "temperature":
        sensor["current_value"] = random.randint(30, 100)
    else:
        if random.random() < 0.3:
            sensor["current_value"] = "YES"
        else:
            sensor["current_value"] = "NO"


# 2. Process the readings: read every sensor and check it with the Step 3 function
def process_readings(sensors, mode):
    for sensor in sensors:
        generate_reading(sensor)
        alert = check_sensor(sensor, mode)
        if alert != None:
            trigger_alert(alert)
            log_event(alert)


# 3. Trigger an alert: print the [ALERT!] line
def trigger_alert(alert):
    print("[ALERT!] " + alert["severity"] + ": " + alert["category"] + ": " + alert["message"])


# 4. Log the event: print the [LOG] line and save it in a list
def log_event(alert):
    print("[LOG] [" + alert["time"] + "] Sending notification to homeowner...")
    event_log.append(alert)


# Check: run once in AWAY and once in HOME
print("--- AWAY ---")
process_readings(sensors, "AWAY")

print("--- HOME ---")
process_readings(sensors, "HOME")

--- AWAY ---
[ALERT!] HIGH: SECURITY: Living Room activity while in AWAY mode!
[LOG] [17:40:07] Sending notification to homeowner...
[ALERT!] HIGH: SECURITY: Front Door activity while in AWAY mode!
[LOG] [17:40:07] Sending notification to homeowner...
--- HOME ---
[ALERT!] LOW: COMFORT: Kitchen temperature is 50F
[LOG] [17:40:07] Sending notification to homeowner...


In [59]:
class Sensor:
    def __init__(self, id, location, type):
        self.id = id
        self.location = location
        self.type = type
        self.current_value = None

    # read(): make a new reading (same as generate_reading in Step 4)
    def read(self):
        if self.type == "temperature":
            self.current_value = random.randint(30, 100)
        else:
            if random.random() < 0.3:
                self.current_value = "YES"
            else:
                self.current_value = "NO"

    # isAbnormal(): check the reading (same rules as check_sensor in Step 3)
    def isAbnormal(self, mode):
        value = self.current_value
        time = datetime.datetime.now().strftime("%H:%M:%S")

        if self.type == "motion" or self.type == "door":
            if value == "YES" and mode == "AWAY":
                return {"time": time, "severity": "HIGH", "category": "SECURITY",
                        "message": self.location + " activity while in AWAY mode!"}

        if self.type == "smoke":
            if value == "YES":
                return {"time": time, "severity": "CRITICAL", "category": "SAFETY",
                        "message": "Smoke in " + self.location + "!"}

        if self.type == "temperature":
            if value < FREEZE_TEMP or value > HOT_TEMP:
                return {"time": time, "severity": "HIGH", "category": "SAFETY",
                        "message": self.location + " temperature is " + str(value) + "F!"}
            if value < COMFORT_MIN or value > COMFORT_MAX:
                if mode == "HOME":
                    return {"time": time, "severity": "LOW", "category": "COMFORT",
                            "message": self.location + " temperature is " + str(value) + "F"}

        return None

    # reset(): clear the value
    def reset(self):
        self.current_value = None


# Update the code: sensors are now Sensor objects instead of dictionaries
sensors = [
    Sensor(1, "Living Room", "motion"),
    Sensor(2, "Front Door", "door"),
    Sensor(3, "Kitchen", "temperature"),
    Sensor(4, "Bedroom", "smoke"),
]

# Update process_readings to use the methods
def process_readings(sensors, mode):
    for sensor in sensors:
        sensor.read()
        alert = sensor.isAbnormal(mode)
        if alert != None:
            trigger_alert(alert)
            log_event(alert)


# Check: create one Sensor object, call read(), and print its value
test_sensor = Sensor(3, "Kitchen", "temperature")
test_sensor.read()
print(test_sensor.current_value)

70


In [65]:
# Update trigger_alert so the [ALERT!] line has the icon, like the sample output
def trigger_alert(alert):
    if alert["category"] == "SECURITY":
        icon = "🚨"
    elif alert["severity"] == "CRITICAL":
        icon = "🔥"
    elif alert["category"] == "SAFETY":
        icon = "⚠️"
    else:
        icon = "💡"
    print("[ALERT!] " + icon + " " + alert["severity"] + ": " + alert["category"] + ": " + alert["message"])


# The 3 "minutes"
times = ["14:30:00", "14:31:00", "14:32:00"]

# Header and mode
print("=== HomeGuard Security System ===")
print("Mode: " + system_mode)

# Main loop: read all sensors 3 times
for time in times:
    print()
    print("Time: " + time)

    for sensor in sensors:
        sensor.read()

        # Print the [READING] line like the sample output
        if sensor.type == "motion":
            if sensor.current_value == "YES":
                print("[READING] " + sensor.location + " Motion: Activity detected")
            else:
                print("[READING] " + sensor.location + " Motion: No activity")

        if sensor.type == "door":
            if sensor.current_value == "YES":
                print("[READING] " + sensor.location + ": OPENED")
            else:
                print("[READING] " + sensor.location + ": CLOSED")

        if sensor.type == "temperature":
            if sensor.current_value < COMFORT_MIN or sensor.current_value > COMFORT_MAX:
                status = "Abnormal"
            else:
                status = "Normal"
            print("[READING] " + sensor.location + " Temperature: " + str(sensor.current_value) + "°F (" + status + ")")

        if sensor.type == "smoke":
            if sensor.current_value == "YES":
                print("[READING] " + sensor.location + " Smoke: DETECTED")
            else:
                print("[READING] " + sensor.location + " Smoke: CLEAR")

        # Check the sensor: event + mode = alert or not
        alert = sensor.isAbnormal(system_mode)
        if alert != None:
            alert["time"] = time
            trigger_alert(alert)
            log_event(alert)

=== HomeGuard Security System ===
Mode: AWAY

Time: 14:30:00


AttributeError: 'dict' object has no attribute 'read'